In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))
from supervisor_agent.state import AppState
from langchain.chat_models import init_chat_model
from langchain.agents.middleware import PIIMiddleware
from langchain.agents import create_agent

llm=init_chat_model(model_provider="groq", model="openai/gpt-oss-120b")

agent=create_agent(
    model=llm,
    middleware=[
        PIIMiddleware("credit_card", strategy="redact", apply_to_output=True),
        # PIIMiddleware("email", strategy="redact")
    ]
)

def gaurdrails_node(appstate:AppState):
    response_text = appstate["final_response"]
    response=agent.invoke({"messages":[response_text]})
    return {"final_response": response["messages"][-1].content}